In [1]:
from pathlib import Path
import os
import sys

project_root = Path.cwd().resolve()
while not (project_root / "src").is_dir() and project_root.parent != project_root:
    project_root = project_root.parent

if not (project_root / "src").is_dir():
    raise RuntimeError("Could not locate the project root.")

os.chdir(project_root)
sys.path.insert(0, str(project_root))

In [2]:
from src.core.config import Config
from src.document.pdf_parser import parse_pdf_document
from src.document.word_parser import parse_word_document

config = Config()

sections = parse_word_document("data/raw/Report.docx")
print(f"Word: {len(sections)} sections, {sum(s['type'] == 'table' for s in sections)} tables")
for section in sections[:5]:
    print(f"  [{section['type']}] {section['title'][:50]} — {len(section['content'])} chars")

sections = parse_pdf_document("data/raw/HuMengqing.pdf", config)
print(f"PDF: {len(sections)} sections, {sum(s['type'] == 'table' for s in sections)} tables")
for section in sections[:5]:
    print(f"  [{section['type']}] {section['title'][:50]} — page {section['page']} — {len(section['content'])} chars")



Word: 62 sections, 5 tables
  [text] Introduction — 3002 chars
  [text] TABLE OF CONTENTS — 1735 chars
  [text] List of Figures — 2126 chars
  [text] List of table — 149 chars
  [text] LIST OF ABBREVIATIONS AND SYMBOLS — 22 chars
PDF: 61 sections, 8 tables
  [text] Learning on OCT-data — page -9 — 408 chars
  [text] SELBSTSTÄNDIGKEITSERKLÄRUNG — page -7 — 458 chars
  [text] ABSTRACT — page -6 — 1282 chars
  [text] CONTENTS — page -5 — 5889 chars
  [text] LIST OF FIGURES — page -3 — 4828 chars


In [3]:
import pdfplumber
from collections import Counter

with pdfplumber.open("data/raw/HuMengqing.pdf") as pdf:
    size_counter = Counter()
    size_samples = {}

    for page in pdf.pages[9:]:  # Skip the first 9 introductory pages
        for char in page.chars:
            size = round(char["size"], 1)
            size_counter[size] += 1
            # Record only one sample of text for each font size
            if size not in size_samples:
                # Collect text from the same line as samples 
                size_samples[size] = ""
            if len(size_samples[size]) < 60:
                size_samples[size] += char["text"]

    print(f"{'Size':>6} | {'Count':>7} | Sample Text")
    print("-" * 60)
    for size, count in size_counter.most_common():
        sample = size_samples[size].strip()[:50]
        print(f"{size:>6} | {count:>7} | {sample}")

  Size |   Count | Sample Text
------------------------------------------------------------
  10.6 |  100856 | IX Abbreviations AM Additive Manufacturing OCT Opt
  10.0 |    2699 | Figure 2.1 Schematic of a Generic Fiber-optic OCT 
  11.0 |    1150 | 2.1 Optical Coherence Tomography  2.2 Artificial N
  12.0 |     454 | LIST OF ABBREVIATIONS AND SYMBOLS 1 Introduction a
   7.6 |      89 | 0𝑁𝑁𝑁𝑁−1𝑁𝑁𝑁𝑁𝑁𝑁𝑁𝑁𝑁𝑁−1𝑁𝑁−𝑥𝑥−1𝑁𝑁𝑁𝑁𝑁𝑁𝑁𝑁𝑁𝑁𝑛𝑛𝑛𝑛𝑁𝑁𝑛𝑛𝑛𝑛𝑛𝑛𝑁𝑁


In [4]:
from src.core.config import Config

from src.document.word_parser import parse_word_document
sections = parse_word_document("data/raw/Report.docx")
print(f"Total: {len(sections)} sections")
print()
for s in sections:
    title = s["title"][:60]
    chars = len(s["content"])
    print(f"  {chars:>6} chars | [{s['type']:>5}] {title}")

Total: 62 sections

    3002 chars | [ text] Introduction
    1735 chars | [ text] TABLE OF CONTENTS
    2126 chars | [ text] List of Figures
     149 chars | [ text] List of table
      22 chars | [ text] LIST OF ABBREVIATIONS AND SYMBOLS
    3689 chars | [ text] Introduction
    2314 chars | [ text] Additive Manufacturing
    2648 chars | [ text] Optical Coherence Tomography
    1019 chars | [ text] Artificial Neural Networks
     616 chars | [ text] Neuron
     774 chars | [ text] Activation Function
     426 chars | [ text] Weights and Biases
     549 chars | [ text] Layers
     632 chars | [ text] Loss Function
     710 chars | [ text] Optimizer
     519 chars | [ text] Gradient Descent
    1019 chars | [ text] Backpropagation
    1473 chars | [ text] Convolutional Neural Networks
    1010 chars | [ text] Convolutional Layer
    1284 chars | [ text] Pooling Layer
    1365 chars | [ text] Flatten Layer
    1146 chars | [ text] Fully Connected Layer
    1343 chars | [ text] Dropout


In [5]:
from src.core.config import Config
from src.document.pdf_parser import parse_pdf_document

config = Config()
sections = parse_pdf_document("data/raw/HuMengqing.pdf", config)

print(f"Total: {len(sections)} sections")
print()

for section in sections:
    title = section["title"][:60]
    character_count = len(section["content"])
    page_number = section.get("page", "?")

    print(
        f"  page {page_number:>3} | {character_count:>6} chars | "
        f"[{section['type']}] {title}"
    )

Total: 61 sections

  page  -9 |    408 chars | [text] Learning on OCT-data
  page  -7 |    458 chars | [text] SELBSTSTÄNDIGKEITSERKLÄRUNG
  page  -6 |   1282 chars | [text] ABSTRACT
  page  -5 |   5889 chars | [text] CONTENTS
  page  -3 |   4828 chars | [text] LIST OF FIGURES
  page  -1 |    980 chars | [text] LIST OF TABLES
  page   0 |    729 chars | [text] LIST OF ABBREVIATIONS AND SYMBOLS
  page   1 |   3852 chars | [text] 1 Introduction and Motivation
  page   3 |    650 chars | [text] 2 Theoretical Basis and Current Situation
  page   3 |   2663 chars | [text] 2.1 Optical Coherence Tomography
  page   4 |   6533 chars | [text] 2.2.1 Feed-forward Neural Networks
  page   8 |   3484 chars | [text] 2.2.2 Convolutional Neural Networks
  page  10 |   1340 chars | [text] 2.2.3.1 LeNet5
  page  11 |   1457 chars | [text] 2.2.3.2 AlexNet
  page  12 |   1112 chars | [text] 2.2.3.3 VGG
  page  12 |   4390 chars | [text] 2.2.3.4 ResNet
  page  16 |   2418 chars | [text] 2.2.3.5 EfficientNe

In [6]:
import json
from pathlib import Path

from src.core.config import Config
from src.document.pdf_parser import parse_pdf_document
from src.document.word_parser import parse_word_document

output_directory = Path("output/sections")
output_directory.mkdir(parents=True, exist_ok=True)

documents = {
    Path("data/raw/Report.docx"): parse_word_document,
    Path("data/raw/HuMengqing.pdf"): lambda document_path: parse_pdf_document(
        document_path, Config()
    ),
}

for document_path, parser in documents.items():
    sections = parser(document_path)
    output_path = output_directory / f"{document_path.stem}.json"
    output_path.write_text(
        json.dumps(sections, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(f"Saved {len(sections)} sections to {output_path}")


Saved 62 sections to output/sections/Report.json
Saved 61 sections to output/sections/HuMengqing.json


In [7]:
import json
from pathlib import Path

from src.core.config import Config
from src.document.chunker import chunk_sections
from src.document.pdf_parser import parse_pdf_document
from src.document.word_parser import parse_word_document

output_directory = Path("output/chunks")
output_directory.mkdir(parents=True, exist_ok=True)
config = Config()

documents = {
    Path("data/raw/Report.docx"): parse_word_document,
    Path("data/raw/HuMengqing.pdf"): lambda document_path: parse_pdf_document(
        document_path, config
    ),
}
all_chunks = []

for document_path, parser in documents.items():
    sections = parser(document_path)
    chunks = chunk_sections(sections, config)
    all_chunks.extend(chunks)
    output_path = output_directory / f"{document_path.stem}_chunks.json"
    output_path.write_text(
        json.dumps(chunks, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(f"Saved {len(chunks)} chunks to {output_path}")

all_chunks_path = output_directory / "all_chunks.json"
all_chunks_path.write_text(
    json.dumps(all_chunks, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"Saved {len(all_chunks)} chunks to {all_chunks_path}")


/Users/humengqing/Documents/Code/VSCode/doc-qa-agent/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Saved 88 chunks to output/chunks/Report_chunks.json
Saved 99 chunks to output/chunks/HuMengqing_chunks.json
Saved 187 chunks to output/chunks/all_chunks.json
